# Module 3.1: Answer Hotel Questions and Record Booking Requests

Use a fixed hotel search and a protected reservation command with your configured **Neo4j database** and **Amazon Bedrock**. This notebook creates no AWS resources.

This notebook retrieves hotel facts, responds when the graph has no answer, enforces Neo4j's maximum-guests rule, and safely retries a reservation request.

**Overview**

- **Grounded answer:** The agent answers using only hotel facts returned from Neo4j.
- **Tool choice:** The agent reads two read-tool descriptions and picks the one that fits each question.
- **Hybrid search:** One search combines vector and full-text matching.
- **Idempotent request:** A request can run again without creating a duplicate reservation.

## Neo4j and AWS roles

| Neo4j owns | AWS owns |
|---|---|
| Connected hotel knowledge: hotels, amenities, ratings, policies | Amazon Bedrock reasons over the retrieved context |
| The vector index `hotel_chunk_embeddings` and full-text index `hotel_chunk_fulltext` | Amazon Nova 2 creates the query embedding |
| The reviewed Cypher traversal that enriches a matched `Chunk` with its hotel |  |
| The maximum-guests rule and the idempotent `ReservationRequest` write |  |

The notebook uses one fixed `HybridCypherRetriever`. It uses `NAIVE` fusion, `top_k=5`, and one reviewed traversal. It accepts one `query` argument. Module 2 compares retrieval roles and selects this fixed Hybrid-Cypher pattern for the application.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("03-grounded-booking-agent")
print(f"Workshop root: {REPO_ROOT}")


In [ ]:
import inspect
import json
import os
import uuid
from datetime import date, timedelta

import boto3

from workshop.agent_tools import PASSAGE_TOOL, READ_TOOLS, RECORD_TOOL
from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id
from workshop.contracts import (
    MAX_GUESTS,
    OVER_LIMIT_GUESTS,
    ReservationReason,
    ReservationStatus,
)
from workshop.fixtures import (
    HERO_NAME,
    HERO_SOURCE,
    apply_reservation_fixtures,
    load_manifest,
    readiness_problems,
)
from workshop.grounding import MISSING_LIVE_AVAILABILITY
from workshop.hybrid_retrieval import Neo4jConfig, search_hotel_knowledge
from workshop.prompts import BASE_GROUNDING_PROMPT
from workshop.retrieval_setup import report_problems
from workshop.workshop_utils import (
    ToolTraceHook,
    selected_tool_names,
    show_result,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = configure_aws_region()
MODEL_ID = default_model_id()
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured, so live cells will be skipped. Set NEO4J_URI/USERNAME/PASSWORD/DATABASE.")
if not BEDROCK_READY:
    print("AWS credentials are not configured, so live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to retrieve the fixture hotel.")

## 1. Prepare the graph for hotel search and booking

Run the cell below to set up the Module 3 graph data. It applies fixture hotel IDs, uniqueness constraints, and the maximum-guests rule. You can run it more than once. It also checks that both retrieval indexes are online and the example hotel is present. It does not change the canonical hotel facts.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_reservation_fixtures(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

## 2. Search for one hotel's amenities and rating

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

Run this search to find the hotel's amenities and guest rating. Full-text search matches the hotel name. Vector search matches the request for amenities and a rating. The reviewed traversal returns the connected hotel, its amenities, its rating, and the stable `hotel_id`. The results are the highest-scoring grounded matches.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping hotel details question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']} | hotel_id={top['hotel_id']}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Amenities: {', '.join(top['amenities'])}")
    print("\nSource chunk text (the provenance for these facts):")
    print(top["chunk_text"][:600])

### How the search finds hotel facts

- **Vector matching:** It finds text with the same meaning as "amenities and guest rating".
- **Full-text matching:** It finds the exact hotel name and location terms.
- **Reviewed Cypher traversal:** It follows the matched `Chunk` to its hotel. It returns up to 12 connected amenities, the guest rating, and the opaque `hotel_id`. The reservation command uses this `hotel_id` to identify the hotel.

This notebook fixes the fusion behavior and `top_k`. The retrieval contract stays exact across runs. Module 2 compares the retrieval roles and chooses this configuration for the application. The graph fields come from the data extracted into Neo4j. The returned source `Chunk` and provenance show where those fields came from. Model wording can vary between runs. The retrieval contract stays the same each time.

## 3. Register two read tools and let the agent choose

The agent gets two ways to read the graph, and it decides which one each question needs.

- **`search_hotel_passages`:** Hybrid search over hotel text. It returns up to five passages and the hotel facts linked to them.
- **`query_hotel_records`:** Text2Cypher over the stored hotel records. It returns the rows a generated read-only Cypher query produced.

**The agent loop**

1. You ask a question.
2. The model reads every tool specification: the name, the description, and the input schema.
3. The model chooses a tool and supplies its input.
4. Strands runs the tool and hands the result back to the model.
5. The model writes its answer from what came back.

Nothing forces a tool call. The system prompt says that a hotel fact needs tool evidence, and each tool description says which questions belong to it. A turn that needs no hotel fact gets a direct reply.

### Strands agent basics

**Brief overview**

- **`Agent`:** It sends the question to the model and runs the tools the model asks for.
- **`BedrockModel`:** It connects the agent to the Amazon Bedrock model named by the model ID.
- **`@tool`:** It turns a Python function into a tool specification the model can read.
- **`ToolTraceHook`:** It prints each tool call as it happens and records what came back.

In [ ]:
print(f"The model receives {len(READ_TOOLS)} tool specifications.\n")
for read_tool in READ_TOOLS:
    print(json.dumps(read_tool.tool_spec, indent=2))
    print()

print("The wrapper behind the first specification:\n")
print(inspect.getsource(READ_TOOLS[0]))

### What the model actually reads

The printed specification is all the model gets: a name, a description, and an input schema. The docstring supplies most of it, so read the specification rather than the docstring to know what was sent. The opening paragraphs of each description carry the routing rule, and each one names the other tool at its boundary.

Both wrappers reject an empty or whitespace-only query when they run. The generated schema can require a `query` string. It cannot say that the string has to contain something.

Both wrappers also return the same envelope: `ok`, the evidence they found, and a `grounding_result` verdict of `answerable` and `missing_fact`. The verdict is a property of the question against this graph, so it is the same two fields whichever tool ran.

### The second model call inside the structured tool

`query_hotel_records` asks a model to write Cypher, so one question can reach a model twice.

1. The agent's model chooses `query_hotel_records` and passes the question through.
2. Inside the tool, Text2Cypher asks a model to write one Cypher query against the pinned graph schema.
3. Neo4j plans that query with `EXPLAIN` and runs it only when the plan is read-only.
4. The tool returns the generated Cypher beside the rows it produced.

The Cypher comes back so you can read it. A query that runs is not the same as a query that asked the right question, so treat it as evidence to inspect rather than as proof.

### Empty rows and failed queries mean different things

- **Empty `records`:** The query ran and matched nothing. That is a successful read. It does not prove the graph lacks the fact, because a generated query can also ask the wrong question.
- **An error result:** The query could not be generated, was refused because its plan was not read-only, or was rejected by the database. The tool returns `ok: false` with an error code, and the trace shows the status as an error.

In [ ]:
from strands import Agent
from strands.models import BedrockModel


def build_agent():
    """Build one fresh agent and its own trace.

    A fresh agent per question is what makes the routing examples honest. An
    agent that already answered a passage question carries that exchange in
    its messages, and the next choice is then partly an echo of the last one.
    The trace is per agent for the same reason: recorded calls from an earlier
    question have no business in a later check.
    """
    trace = ToolTraceHook()
    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=list(READ_TOOLS),
        system_prompt=BASE_GROUNDING_PROMPT,
        hooks=[trace],
    )
    return agent, trace


def recorded_payloads(trace):
    """Return the complete JSON payload of every tool call a trace recorded."""
    return [
        payload
        for call in trace.calls
        for payload in call["payloads"]
        if isinstance(payload, dict)
    ]


print("System prompt sent with every question:\n")
print(BASE_GROUNDING_PROMPT)

## 4. Watch the agent route four questions

Each question gets a fresh agent and a fresh trace. Read the trace line under each question: it names the tool the model picked and shows what the tool returned.

| Question | Expected tool |
|---|---|
| Amenities and guest rating for one named hotel | `search_hotel_passages` |
| Average guest rating of the hotels in Paris | `query_hotel_records` |
| Count of hotels that offer a spa | `query_hotel_records` |
| The recorded wording of a cancellation policy | `search_hotel_passages` |

Two of these questions, the Paris average and the spa count, also appear as worked examples in the Text2Cypher prompt inside `query_hotel_records`. That makes the structured path easier for them than it will be for a question the prompt has never seen. They are here because they show the routing rule cleanly, not because they are hard. Swap in a structured question of your own to see how routing holds up without that head start.

Model routing can vary between runs. A mismatch is a signal to improve a tool name or description, so this cell reports what happened and keeps going.

In [ ]:
ROUTING_CASES = (
    (HERO_QUESTION, PASSAGE_TOOL),
    ("What is the average guest rating of the hotels in Paris?", RECORD_TOOL),
    ("How many hotels offer a spa?", RECORD_TOOL),
    (
        f"What is the cancellation policy at {HERO_NAME}? "
        "Quote the recorded wording.",
        PASSAGE_TOOL,
    ),
)

if not RETRIEVAL_READY:
    print("Skipping the routing table: retrieval is not configured.")
else:
    matched = 0
    for question, expected in ROUTING_CASES:
        agent, trace = build_agent()
        print(f"\nQ: {question}")
        routing_result = agent(question)
        chosen = selected_tool_names(routing_result)
        show_result(routing_result)
        if expected in chosen:
            matched += 1
            print(f"   ✅ expected {expected}, used {chosen}")
        else:
            print(f"   ⚠️  expected {expected}, used {chosen or 'no tool'}")

    print(f"\n{matched} of {len(ROUTING_CASES)} questions reached the expected tool.")
    print("A mismatch is a tool description to improve, not a broken notebook.")

## 5. Ask a question the graph cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph stores hotel knowledge and total room capacity. It stores no live room inventory. Either read tool can run and return real hotel evidence, and neither one can confirm availability.

The tool result carries that decision as data. Its `grounding_result` reports `answerable: false` with `missing_fact: live_room_availability`. The check below reads that verdict rather than grading the model's wording, so it detects a fabricated answer during the lab without depending on which phrase the model chose. Either read path may be the one that recognizes the limit, so this question is not part of the routing score above.

In [ ]:
if not RETRIEVAL_READY:
    print("Skipping the availability question: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {AVAILABILITY_QUESTION}")
    availability_result = agent(AVAILABILITY_QUESTION)
    show_result(availability_result)

    verdicts = [
        payload["grounding_result"]
        for payload in recorded_payloads(trace)
        if isinstance(payload.get("grounding_result"), dict)
    ]
    print(f"\nTools used: {selected_tool_names(availability_result) or 'none'}")
    print(f"Verdicts returned: {json.dumps(verdicts)}")

    problems = []
    if not trace.calls:
        problems.append("the agent answered an availability question with no tool call")
    if not any(
        verdict.get("missing_fact") == MISSING_LIVE_AVAILABILITY
        for verdict in verdicts
    ):
        problems.append(
            f"no tool result reported missing_fact={MISSING_LIVE_AVAILABILITY}"
        )
    report_problems(problems, "a tool ran and reported live room availability as unsupported.")

## 6. Answer a turn that needs no tool

> **thanks, that is all**

Nothing in this agent forces retrieval, so a turn with no hotel fact in it should get a direct reply and no tool call. The check reads the tools the turn used out of the turn's own metrics rather than out of the printed trace.

In [ ]:
SOCIAL_TURN = "thanks, that is all"

if not RETRIEVAL_READY:
    print("Skipping the social turn: retrieval is not configured.")
else:
    agent, trace = build_agent()
    print(f"Q: {SOCIAL_TURN}")
    social_result = agent(SOCIAL_TURN)
    show_result(social_result)

    used = selected_tool_names(social_result)
    if used:
        print(f"   ⚠️  used {used}; a thank-you needs no hotel fact")
    else:
        print("   ✅ no tool call: the model answered without reading the graph.")

## 7. Reject an over-limit reservation request

The Neo4j maximum-guests rule sets a 10-guest limit. The command reads and enforces this rule inside the write transaction. It rejects a request for 15 guests before it creates a node.

These cells require your Aura connection through `NEO4J_READY`. They do not use Bedrock. The local fixture manifest supplies the example `hotel_id`, so this write example does not need live search results. The notebook calculates dates from the current day to keep the example valid.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id from fixture manifest: {hero_id}")
    print(f"Caller-created request_id for retries: {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

    assert rejected["status"] == ReservationStatus.REJECTED.value, rejected
    assert rejected["reason_code"] == ReservationReason.MAX_GUESTS_EXCEEDED.value, rejected
    assert rejected["hotel_id"] == hero_id, rejected
    assert rejected["max_guests"] == MAX_GUESTS, rejected

## 8. Create one reservation and safely retry it

Submit a request within the 10-guest limit, then submit it again with the same `request_id`. The first request creates one `ReservationRequest` and links it to the hotel with a `FOR_HOTEL` relationship. The second request finds the existing request and returns `duplicate=true` with its original `created_at`. The uniqueness constraint prevents a second node with the same `request_id`.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    assert accepted["status"] == ReservationStatus.ACCEPTED.value, accepted
    assert accepted["hotel_id"] == hero_id, accepted
    assert accepted["duplicate"] is False, accepted

    assert replay["status"] == ReservationStatus.ACCEPTED.value, replay
    assert replay["hotel_id"] == hero_id, replay
    assert replay["duplicate"] is True, replay

## 9. Verify the reservation in Neo4j

Use the stable `request_id` to confirm that the graph has one accepted request linked to one hotel.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

## Continue to the next modules

- **Module 4:** It deploys the retrieval functions through Amazon Bedrock AgentCore Gateway and AWS Lambda.
- **Module 5:** It packages a deployment-oriented version of the agent for AgentCore Runtime with Docker, Secrets Manager, and IAM boundaries.
- **Module 6:** It adds actor-scoped, cross-session graph memory with provenance and direct correction.

All steps above run against your own Aura instance and Amazon Bedrock. They create no AWS resources.